**2. MOCO TRAINING**

In [ ]:
import digitalhub as dh
import pandas as pd
import matplotlib.pyplot as plt

NOME_PROGETTO = "floods"
project = dh.get_project(NOME_PROGETTO)
print(f"Progetto: {project.name}")

**SETUP PARAMETERS**

In [ ]:
job_name = "check_v1"                                     
dataset = "Test"                                                                       
weights_encoder_sar = "check_sar_v1"     
weights_encoder_opt = "check_opt_v1"                                        

parametri = {
    "job_name": job_name,
    "dataset": dataset,
    "weights_encoder_sar": weights_encoder_sar,
    "weights_encoder_opt": weights_encoder_opt,
    "resume": False, 
    "time_debug": True,   

    "epochs": 200, 
    "batch_size": 16,
    "lr": 0.03,
    "weight_decay": 1e-4,
    "momentum": 0.9,
    "patch_size": 256,
    "n_images1": 4, "n_channels1": 2,              
    "n_images2": 4, "n_channels2": 10,              
    "moco_dim": 128,
    "moco_k": 4096,                                     
    "moco_t": 0.07,
    "symmetric": False,
    "workers": 0,
    "patience": 20,
    "min_delta": 1e-4,                                   
}

volumi = [
    {
        "volume_type": "ephemeral",
        "name": "volume-spazio-dati",
        "mount_path": "/data",      
        "spec": {"size": "200Gi"}   
    }
]

**BUILD ENVIRONMENT**

In [ ]:
moco_1D_train_func = project.new_function(
    name= f'moco_1D_{job_name}',
    kind="python",
    python_version="PYTHON3_10",
    code_src="../src/", 
    handler="train_moco_1D", 
    base_image="pytorch/pytorch:2.1.2-cuda11.8-cudnn8-runtime",
    requirements=["pandas==2.3.3", "numpy==1.26.4", "rasterio==1.4.4", "tqdm==4.70.0", "tifffile==2024.8.30", "torch==2.1.2", "matplotlib==3.10.9", "digitalhub==0.15.11", "digitalhub-runtime-python==0.15.2"]
)

build = moco_1D_train_func.run("build", wait=True)
print(f"BUILD: {build.status.state}")

**TRAINING**

In [ ]:
run_moco_1D = moco_1D_train_func.run(
    action="job", 
    parameters=parametri, 
    volumes=volumi, 
    profile="1xv100-shared",
    local_execution= False,                       
    wait=True
)

print(f"Run moco avviato: {run_moco_1D.id}")

**PLOTS**

In [ ]:
path = project.get_artifact(f"moco_1D_metrics_{job_name}").download(overwrite=True)
df = pd.read_csv(path)

plt.figure(figsize=(8, 5))
plt.plot(df['epoch'], df['train_loss'], color='green', label='Train Loss')
plt.title(f'MoCo 1D- {job_name}')
plt.xlabel('Epochs')
plt.ylabel('Loss InfoNCE')
plt.grid(True)
plt.legend()
plt.show()